# The target and the forced response

Every member of the ensemble follows the same forcing path by calendar year. Before a model can read ENSO, that shared path has to come out
— fit on the training members only — and "wet" has to be defined against one threshold.


## Overview

1. Name the test members, per forcing half, before any statistic is fit.
2. Remove the forced response with a curve fit on the training members only, and check the removal with a test that can fail.
3. Define "wet" against one pooled threshold, and read the base rate per half.
4. Look at the target against the input — the straight-line relation, and the wet rate by tenth of the input — and note the inputs not
   chosen.

Shared code lives in `lensml.py` beside the chapters (parameters, transforms, ladder, scoring, paired test, register); `show()` prints a
function's source where a chapter introduces it.


## Prerequisites

| Concepts | Importance | Notes |
|---|---|---|
| [Xarray tutorial](https://tutorial.xarray.dev/) | Necessary | a Zarr store, a box selection, `groupby` |
| [Chapter 1](01-find-the-data.ipynb) | Helpful | the question and the cache |
| Polynomial fits, quantiles, a binomial standard error | Helpful | `np.polyfit`, `Series.quantile`, √(p(1−p)/n) |

- **Time to learn**: about 20 minutes; every cell runs in seconds from the cache.
- **System requirements**: the `environment.yml` at the repository root; no account, no local data beyond the committed cache.


## Imports


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lensml import (
    ANOMALIES,
    SEASON,
    apply_transforms,
    cite,
    decile_edges,
    draw_test_members,
    fit_transforms,
    load_cache,
    show,
    wet_by_bin,
)
from scipy.stats import norm

## Name the test members first

`TEST_FRACTION` of each half's members are drawn, by member, from a `RandomState` created from `RNG_SEED` right before the draw, so
re-running the cell redraws the same members. Nothing in this cookbook is fit on them; [chapter 6](06-test-and-justify.ipynb) scores
them. They are not a blind test set, though: during development these same members were scored with two candidate final models
(chapter 6 gives both), so a result on them is provisional, and a fresh test would need data no choice was made on; chapter 7 uses the same members' SSP3-7.0 winters.


In [ ]:
table, cache_header, access_date = load_cache()
show(draw_test_members)

In [ ]:
test_members = draw_test_members(table)
table["split"] = np.where(
    table["member_id"].isin(test_members), "test", "train"
)
is_train = table["split"].eq("train")
print(f"test members (scored in chapter 6): {test_members}")
print(f"training members: {sorted(table.loc[is_train, 'member_id'].unique())}")

## Remove the forced response

The raw SON Niño 3.4 box mean and the winter precipitation both carry a forced trend (Rodgers et al. 2021 §3.1, *Mean state changes*; the check below prints
each one's correlation with year). A degree-`FORCED_DEG` polynomial in year, fit to the *training-member* ensemble mean, is subtracted from
every member alike, leaving each member's own departure from the shared path — the ENSO state, not the decade. Without it, the models would
learn time-of-century and it would read as ENSO skill.

Then the target: `wet` = 1 when the detrended precipitation sits above the `WET_QUANTILE` of the pooled *training* members. **One**
threshold for every member: a per-member quantile would fix each member's base rate by construction, and the per-half check below could
never fail. Curve and threshold are transforms fit on training members and applied to everyone else — as a `StandardScaler` inside a
`Pipeline` is (scikit-learn's *Common pitfalls*, §12.2.1) — written as two functions so that every validation fold in chapters 3–5 refits
them. One curve is fit to both forcing halves together; the per-half check below is what licenses pooling them.


In [ ]:
show(fit_transforms, apply_transforms)

In [ ]:
TRANSFORMS = fit_transforms(table[is_train])  # fit on ALL training members
table = apply_transforms(table, TRANSFORMS)
WET_THRESHOLD = TRANSFORMS["wet_threshold"]
print(
    f"wet threshold (pooled training members) = {WET_THRESHOLD:+.4f} mm/day "
    f"of detrended {SEASON} precipitation"
)

### The check that can fail

A forced path the curve missed (too low a degree, a mid-century bump) survives the subtraction as decades whose mean anomaly sits away
from zero. Each decade's mean anomaly over the training members is divided by its standard error; the removal **passes** only if every
decade of both series lies within `±DECADE_TOL` standard errors. The correlation of the anomaly with year is not used: it is zero by
construction for any `FORCED_DEG >= 1`, so it could never fail. The last line prints the chance that a correct removal trips the tolerance
somewhere by luck.


In [ ]:
DECADE_TOL = 3.0
d = table[is_train]
decade = (d["winter"] // 10) * 10
for anom, raw in ANOMALIES.items():
    r = d[raw].corr(d["winter"])
    print(
        f"corr({raw:20s}, year) = {r:+.3f}   "
        "(the forced trend in the raw series, training members)"
    )
removal_ok, n_tests = True, 0
for anom in ANOMALIES:
    g = d.groupby(decade)[anom]
    z = g.mean() / (d[anom].std() / np.sqrt(g.size()))
    worst = z.abs().idxmax()
    ok = bool((z.abs() <= DECADE_TOL).all())
    removal_ok &= ok
    n_tests += len(z)
    print(
        f"{anom:12s} decade-mean anomaly in standard errors: worst {z[worst]:+.2f} "
        f"(the {worst}s) · tolerance ±{DECADE_TOL:.0f} → {'PASS' if ok else 'FAIL'}"
    )
false_alarm = 1 - (1 - 2 * norm.sf(DECADE_TOL)) ** n_tests
verdict = (
    "PASS"
    if removal_ok
    else "FAIL — a forced path survived; change FORCED_DEG"
)
print(
    f"forced-response removal: {verdict}   ({n_tests} decade means; chance a "
    f"correct removal trips the tolerance somewhere ≈ {false_alarm:.0%})"
)

```{admonition} Try it: make the check fail
:class: tip
Set `FORCED_DEG = 0` in `lensml.py`, restart the kernel and re-run this chapter. Which line flips to `FAIL`, and in which decades? Restore
`FORCED_DEG = 3` before moving on.
```

## The base rate, per half

Read the **base rate** first. With `WET_QUANTILE = 2/3` it sits near one third by design, so an accuracy near two thirds is what "always
dry" scores — which is why this cookbook reports precision and recall against the base rate (scikit-learn's *Classification metrics*
guide). The two forcing halves share one threshold, so their base rates are free to differ: that is the check before pooling them. The
cell then prints, per training member, the correlation of input and target, counts their signs, and divides the mean correlation by its
standard error.


In [ ]:
train_tbl = table[is_train]
base_rate = train_tbl["wet"].mean()
print(
    f"base rate P(wet), training members = {base_rate:.3f}  "
    f"(n = {len(train_tbl)} member-winters)"
)
print(
    "\nper-half base rate (training members) — free to differ because the "
    "threshold is one for both halves:"
)
print(
    train_tbl.groupby("forcing_variant")["wet"].agg(["mean", "size"]).round(3)
)
print("\nper-half detrended precipitation (training members):")
stats = ["mean", "std", "median"]
print(train_tbl.groupby("forcing_variant")["prect_anom"].agg(stats).round(3))


def corr_in(d):
    return d["nino34_anom"].corr(d["prect_anom"])


per_member_corr = train_tbl.groupby("member_id").apply(corr_in)
print("\nper-member correlation(nino34_anom, prect_anom), training members:")
print(per_member_corr.round(3).to_string())
pos, neg = int((per_member_corr > 0).sum()), int((per_member_corr < 0).sum())
n_m, sd_m = len(per_member_corr), per_member_corr.std(ddof=1)
print(
    f"\nper-member sign count: {pos} positive · {neg} negative of {n_m} "
    f"training members · mean {per_member_corr.mean():+.3f} · SD {sd_m:.3f}"
)
by_half = train_tbl.groupby(["forcing_variant", "member_id"]).apply(corr_in)
half_means = by_half.groupby(level=0).mean()
print(
    "per-half mean correlation: "
    + " · ".join(f"{half} {v:+.3f}" for half, v in half_means.items())
)
t_mean = per_member_corr.mean() / (sd_m / np.sqrt(n_m))
print(
    f"mean per-member correlation in standard errors (SD / √{n_m}): {t_mean:+.2f}"
    f" → {'distinguishable' if abs(t_mean) > 2 else 'indistinguishable'} from zero"
    " at 2 standard errors"
)

In [ ]:
d = table[is_train]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].hist(
    d["prect_anom"],
    bins=30,
    color="#C9A23A",
    edgecolor="#C9A23A",
    linewidth=0.5,
)
axes[0].axvline(
    WET_THRESHOLD, lw=1, c="k", ls="--", label="wet threshold (training)"
)
axes[0].legend()
axes[0].set(
    xlabel=f"{SEASON} Front Range precipitation, forced response removed (mm/day)",
    ylabel="member-winters",
    title="Target distribution (training members)",
)
marks = {
    0: ("#1b9e77", "o", "dry"),
    1: ("#e7298a", "^", "wet"),
}  # colour AND shape
for wet, (c, m, lab) in marks.items():
    dd = d[d.wet == wet]
    axes[1].scatter(
        dd["nino34_anom"],
        dd["prect_anom"],
        s=12,
        alpha=0.6,
        c=c,
        marker=m,
        label=lab,
    )
axes[1].axvline(0, lw=0.8, c="k")
axes[1].legend()
axes[1].set(
    xlabel="autumn Niño 3.4 anomaly, forced response removed (K)",
    ylabel=f"{SEASON} precipitation anomaly (mm/day)",
    title="Input vs target",
)
cite(fig, access_date)
plt.tight_layout()

**Reading the straight-line relation.** NWS Boulder's sign for east of the Divide is positive — warmer autumn Pacific, wetter winter —
and stated for late winter and spring, not for the DJF season scored here. Read the `per-member sign count` line: do the members share a
sign? Then the `standard errors` line under it: it says whether the mean correlation is distinguishable from zero. Stagge et al.'s two
regional models set the observed context: "the North–Wyoming model explained 67.1% of winter precipitation" (a model bounded by 41.2° to
45.5° N) and "the South–Sonora dipole model explained less variance (34.5%)". By latitude the Front Range falls in neither, but in the
paper's "transition zone (32–40° N)", east of the "steep eastern slope" along which the northern signal "dissipates abruptly"; and in the
northern pole the sign is reversed, with "increased winter precipitation during La Niña events".

## The wet rate by tenth of the input

A correlation sees only a straight line. The cell below splits the training winters into ten equal-count bins of the Niño 3.4 anomaly
and prints the share that are wet in each, with a binomial standard error and the distance from the base rate in those errors.


In [ ]:
show(wet_by_bin)
edges = decile_edges(
    train_tbl
)  # chapter 6 applies these same edges to the test members
bins, chi2, p_chi2, dof = wet_by_bin(train_tbl, edges)
print(
    f"wet rate by decile of the autumn Niño 3.4 anomaly, training members "
    f"(base rate {base_rate:.3f}):"
)
print(bins.round(3).to_string())
print(
    f"chi-square test of wet against decile, training members: "
    f"chi2 = {chi2:.1f}, dof = {dof}, p = {p_chi2:.2g}"
)

Read the chi-square line first: it says whether the wet rate differs between bins by more than chance, over the training members. Then
the table: the coldest bin (the strongest La Niña autumns) and the warmest bins sit below the base rate, the middle bins above it — the
extremes drier than the middle, a shape a straight line cannot follow, which is why the correlation reads near zero while the bins do not.
Treat any single bin with care: with ten bins, one departure of a few standard errors can be luck. Chapter 6 applies these same edges to
the test members to see which part of the shape repeats.

```{admonition} Try it: may the halves be pooled?
:class: tip
From the per-half tables, argue whether pooling cmip6 and smbb is justified. Then compare the sign you wrote before chapter 1's data cells
with the decile table: which rule — the NWS sign, its reverse, or neither — would the bins reward?
```


## The one input, and the ones not chosen

One input was fixed because the question is about ENSO and a one-input model is one a scientist can read whole. The catalogue offers other
monthly fields in both halves, each a fair input only as an **autumn** value, known before the winter starts.

| Autumn (SON) field | What it stands for | Here |
|---|---|---|
| TS over `NINO34` | the ENSO state | the one input (`FEATURES`) |
| PSL or Z3 over the North Pacific | a stand-in for the autumn PNA state | the strongest next candidate |
| TREFHT over `FRONT_RANGE` | the box's own autumn temperature | weaker: the box's state, not the teleconnection |
| any field averaged over the winter months | the outcome's own season | never: known only after the outcome |
| `forcing_variant` | which half a member belongs to | not an input; the per-half check decides pooling |

A new input means a new box mean in `derive_member_winters` and a rebuild, so it is a costed next change, never a quiet edit to `FEATURES`.
NWS Pueblo's outlook names PNA and NAO as the patterns with the "highest correlations wet/dry".


## Summary

Test members were named before anything was fit. The forced response was removed with a curve fit on the training members only, and a
decade-by-decade check that can fail says whether it worked. "Wet" is one pooled threshold, so the per-half base rates are a real check.
The straight-line relation is weak; over the training members the decile table shows a shape a line misses, which chapter 6 checks on
the test members.

### What's next?

[Chapter 3](03-split-by-member.ipynb) builds the member-grouped folds and measures what a member-blind split, or transforms fit before the
split, would do to the score.

## Resources and references

| Resource | Where it bears | Licence |
|---|---|---|
| Rodgers, K. B. et al. (2021). *Earth System Dynamics* 12, 1393–1411. https://doi.org/10.5194/esd-12-1393-2021 | the forced response in the ensemble | CC BY 4.0 |
| scikit-learn User Guide, §12 Common pitfalls and recommended practices; §3.4 Metrics and scoring. https://scikit-learn.org/stable/ | transforms fit on training data only; the base rate | BSD-3-Clause |
| Stagge, J. H. et al. (2023). Orographic amplification of El Niño teleconnections on winter precipitation across the Intermountain West. *Nature Water* 1, 1016–1026. https://doi.org/10.1038/s44221-023-00163-9 | the two poles | CC BY 4.0 |
| NWS Boulder (2020). La Niña and the Upcoming 2020-2021 Winter Season. https://www.weather.gov/media/bou/LaNinaWinterOutlook_Sept20.pdf | the expected sign and its season | US Government |
| NWS Pueblo (2025). Winter of 2025-2026 Outlook. https://www.weather.gov/media/pub/pdf/SC_SE_CO_Winter_2025_26_Outlook.pdf | PNA and NAO | US Government work |
